# Gaussian Thompson Sampling for Commodity Allocation

**Goal:** Apply Thompson Sampling to real commodity returns using Gaussian posteriors (not Bernoulli).

**Time:** 15 minutes

**What you'll learn:**
- How to extend Thompson Sampling from Bernoulli to Gaussian rewards
- Normal-Normal conjugacy for continuous returns
- Backtest a Thompson Sampling allocation strategy on WTI, Gold, and Corn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from scipy.stats import norm
from datetime import datetime, timedelta

sns.set_style('whitegrid')
np.random.seed(42)

## Load Real Commodity Data

We'll use ETFs as proxies: USO (oil), GLD (gold), CORN (agriculture).

In [ ]:
# Download 3 years of daily data
tickers = ['USO', 'GLD', 'CORN']
commodity_names = ['WTI Crude', 'Gold', 'Corn']

end_date = datetime.now()
start_date = end_date - timedelta(days=3*365)

print("Downloading commodity data...")
data = yf.download(tickers, start=start_date, end=end_date, progress=False)['Adj Close']

# Compute weekly returns (resample to weekly, then pct_change)
weekly_prices = data.resample('W').last()
weekly_returns = weekly_prices.pct_change().dropna()

print(f"\nLoaded {len(weekly_returns)} weeks of data")
print(f"Date range: {weekly_returns.index[0].date()} to {weekly_returns.index[-1].date()}")

# Summary statistics
print("\nWeekly return statistics:")
print(weekly_returns.describe())

## Visualize Raw Returns

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

for i, (ticker, name) in enumerate(zip(tickers, commodity_names)):
    axes[i].plot(weekly_returns.index, weekly_returns[ticker], linewidth=1.5, alpha=0.7)
    axes[i].axhline(0, color='black', linestyle='--', alpha=0.3)
    axes[i].set_title(f'{name} ({ticker}) Weekly Returns', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Return', fontsize=11)
    axes[i].grid(alpha=0.3)
    
    # Add mean line
    mean_ret = weekly_returns[ticker].mean()
    axes[i].axhline(mean_ret, color='red', linestyle=':', alpha=0.5, 
                    label=f'Mean: {mean_ret:.4f}')
    axes[i].legend(fontsize=10)

axes[-1].set_xlabel('Date', fontsize=11)
plt.tight_layout()
plt.show()

print("Observations:")
print("- High volatility (commodity returns are noisy)")
print("- Means close to zero (long-run commodity returns ≈ 0)")
print("- Non-stationary (regimes shift over time)")

## Gaussian Thompson Sampling: Normal-Normal Conjugacy

For Gaussian rewards with known variance, use Normal prior on the mean:

**Prior:** μ ~ N(μ₀, σ₀²)

**Likelihood:** r ~ N(μ, σ²) with known σ²

**Posterior:** μ ~ N(μ₁, σ₁²) where:
- τ₁ = τ₀ + n/σ² (precision update)
- μ₁ = (τ₀·μ₀ + n·r̄/σ²) / τ₁ (precision-weighted mean)
- σ₁² = 1/τ₁

In [ ]:
class GaussianThompsonSampling:
    def __init__(self, n_arms, sigma_obs, mu_0=0.0, sigma_0=0.1):
        """
        Gaussian Thompson Sampling with Normal-Normal conjugacy.
        
        Parameters:
        - n_arms: Number of arms
        - sigma_obs: Known observation standard deviation
        - mu_0: Prior mean
        - sigma_0: Prior standard deviation
        """
        self.n_arms = n_arms
        self.sigma_obs = sigma_obs
        
        # Posterior parameters
        self.mu = np.ones(n_arms) * mu_0
        self.sigma = np.ones(n_arms) * sigma_0
    
    def select_arm(self):
        """Sample from posterior and select best."""
        samples = np.random.normal(self.mu, self.sigma)
        return np.argmax(samples)
    
    def update(self, arm, reward):
        """Update posterior after observing reward."""
        # Precision formulation (easier for Normal-Normal)
        tau_prior = 1 / self.sigma[arm]**2
        tau_obs = 1 / self.sigma_obs**2
        
        tau_post = tau_prior + tau_obs
        mu_post = (tau_prior * self.mu[arm] + tau_obs * reward) / tau_post
        sigma_post = np.sqrt(1 / tau_post)
        
        self.mu[arm] = mu_post
        self.sigma[arm] = sigma_post
    
    def get_posteriors(self):
        """Return current posterior parameters."""
        return self.mu.copy(), self.sigma.copy()

print("GaussianThompsonSampling class defined.")

## Backtest: Thompson Sampling vs Equal-Weight Allocation

Each week, Thompson Sampling picks one commodity to allocate 100% capital. We'll compare to equal-weight (33% each).

In [ ]:
def backtest_thompson(returns_df, sigma_obs=0.05, mu_0=0.0, sigma_0=0.1):
    """
    Backtest Thompson Sampling allocation.
    
    Each week:
    1. Sample from posterior for each commodity
    2. Allocate 100% to commodity with highest sample
    3. Observe return
    4. Update posterior
    """
    n_weeks = len(returns_df)
    n_arms = len(returns_df.columns)
    
    bandit = GaussianThompsonSampling(n_arms, sigma_obs, mu_0, sigma_0)
    
    # Track results
    selections = np.zeros(n_weeks, dtype=int)
    portfolio_returns = np.zeros(n_weeks)
    mu_history = np.zeros((n_weeks + 1, n_arms))
    sigma_history = np.zeros((n_weeks + 1, n_arms))
    
    mu_history[0], sigma_history[0] = bandit.get_posteriors()
    
    for t in range(n_weeks):
        # Select commodity
        arm = bandit.select_arm()
        selections[t] = arm
        
        # Observe return (100% allocated to selected commodity)
        reward = returns_df.iloc[t, arm]
        portfolio_returns[t] = reward
        
        # Update posterior
        bandit.update(arm, reward)
        
        # Track posteriors
        mu_history[t+1], sigma_history[t+1] = bandit.get_posteriors()
    
    return selections, portfolio_returns, mu_history, sigma_history

# Estimate observation noise from data
sigma_obs = weekly_returns.std().mean()
print(f"Using σ_obs = {sigma_obs:.4f} (average weekly volatility)")

# Run backtest
selections, ts_returns, mu_hist, sigma_hist = backtest_thompson(
    weekly_returns, sigma_obs=sigma_obs
)

# Equal-weight benchmark (33% each commodity)
ew_returns = weekly_returns.mean(axis=1).values

print(f"\nBacktest complete over {len(ts_returns)} weeks.")

## Results: Cumulative Performance

In [ ]:
# Compute cumulative returns
ts_cumulative = (1 + ts_returns).cumprod()
ew_cumulative = (1 + ew_returns).cumprod()

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(weekly_returns.index, ts_cumulative, label='Thompson Sampling', linewidth=2)
ax.plot(weekly_returns.index, ew_cumulative, label='Equal-Weight', linewidth=2, linestyle='--')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Cumulative Return (Base = 1.0)', fontsize=12)
ax.set_title('Thompson Sampling vs Equal-Weight Allocation\nWeekly Rebalancing', 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Summary statistics
print("Performance Summary:")
print(f"Thompson Sampling:")
print(f"  Total Return:  {(ts_cumulative[-1] - 1) * 100:6.2f}%")
print(f"  Ann. Return:   {ts_returns.mean() * 52 * 100:6.2f}%")
print(f"  Ann. Vol:      {ts_returns.std() * np.sqrt(52) * 100:6.2f}%")
print(f"  Sharpe:        {ts_returns.mean() / ts_returns.std() * np.sqrt(52):6.2f}")

print(f"\nEqual-Weight:")
print(f"  Total Return:  {(ew_cumulative[-1] - 1) * 100:6.2f}%")
print(f"  Ann. Return:   {ew_returns.mean() * 52 * 100:6.2f}%")
print(f"  Ann. Vol:      {ew_returns.std() * np.sqrt(52) * 100:6.2f}%")
print(f"  Sharpe:        {ew_returns.mean() / ew_returns.std() * np.sqrt(52):6.2f}")

## Allocation Over Time: Which Commodities Were Selected?

In [ ]:
# Compute rolling allocation (100-week window)
window = 100
rolling_alloc = np.zeros((len(selections) - window, 3))

for t in range(window, len(selections)):
    recent = selections[t-window:t]
    for k in range(3):
        rolling_alloc[t-window, k] = (recent == k).mean()

# Plot stacked area
fig, ax = plt.subplots(figsize=(12, 6))

dates = weekly_returns.index[window:]
ax.stackplot(dates, rolling_alloc.T, labels=commodity_names, alpha=0.7)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Allocation (100-week rolling)', fontsize=12)
ax.set_title('Thompson Sampling Allocation Shifts Over Time', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=11)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print("Observations:")
print("- Allocation shifts as posterior beliefs update")
print("- No commodity gets 100% (Thompson Sampling always explores)")
print("- Allocation responds to recent performance (adaptive)")

## Posterior Evolution: Watch Beliefs About Mean Returns

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Plot 1: Posterior means
for i, name in enumerate(commodity_names):
    axes[0].plot(weekly_returns.index, mu_hist[1:, i], label=name, linewidth=2)

axes[0].axhline(0, color='black', linestyle='--', alpha=0.3)
axes[0].set_ylabel('Posterior Mean Return', fontsize=11)
axes[0].set_title('Posterior Mean Evolution', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Plot 2: Posterior standard deviations (uncertainty)
for i, name in enumerate(commodity_names):
    axes[1].plot(weekly_returns.index, sigma_hist[1:, i], label=name, linewidth=2)

axes[1].set_xlabel('Date', fontsize=11)
axes[1].set_ylabel('Posterior Std Dev (Uncertainty)', fontsize=11)
axes[1].set_title('Posterior Uncertainty Evolution', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insights:")
print("- Posterior means fluctuate around zero (long-run commodity mean)")
print("- Uncertainty decreases over time (posteriors concentrate)")
print("- More-selected commodities have lower uncertainty (more data)")

## Modify This: Add Risk Adjustment

Current implementation allocates 100% to highest sampled mean. Adjust for risk by sampling Sharpe ratios instead.

In [ ]:
# EXPERIMENT: Risk-adjusted Thompson Sampling
# Instead of sampling mean returns, sample Sharpe ratios
# Sharpe = μ / σ, where σ is estimated from rolling window

def backtest_thompson_sharpe(returns_df, lookback=52):
    """
    Thompson Sampling with Sharpe ratio instead of raw returns.
    
    For each commodity:
    - Estimate volatility from rolling window
    - Sample μ ~ N(posterior mean, posterior std)
    - Compute sampled Sharpe = μ / σ_rolling
    - Select commodity with highest Sharpe sample
    """
    n_weeks = len(returns_df)
    n_arms = len(returns_df.columns)
    
    # Estimate observation noise
    sigma_obs = returns_df.std().mean()
    bandit = GaussianThompsonSampling(n_arms, sigma_obs)
    
    selections = np.zeros(n_weeks, dtype=int)
    portfolio_returns = np.zeros(n_weeks)
    
    for t in range(n_weeks):
        # Estimate volatility from rolling window
        if t < lookback:
            rolling_vol = returns_df.iloc[:t+1].std().values
        else:
            rolling_vol = returns_df.iloc[t-lookback:t].std().values
        
        # Sample mean returns
        mu_samples = np.random.normal(bandit.mu, bandit.sigma)
        
        # Compute Sharpe samples
        sharpe_samples = mu_samples / (rolling_vol + 1e-6)
        
        # Select arm with highest Sharpe
        arm = np.argmax(sharpe_samples)
        selections[t] = arm
        
        # Observe and update
        reward = returns_df.iloc[t, arm]
        portfolio_returns[t] = reward
        bandit.update(arm, reward)
    
    return selections, portfolio_returns

# Run Sharpe-adjusted version
selections_sharpe, returns_sharpe = backtest_thompson_sharpe(weekly_returns)

# Compare
cumulative_sharpe = (1 + returns_sharpe).cumprod()

plt.figure(figsize=(12, 6))
plt.plot(weekly_returns.index, ts_cumulative, label='Thompson (raw returns)', linewidth=2)
plt.plot(weekly_returns.index, cumulative_sharpe, label='Thompson (Sharpe-adjusted)', linewidth=2)
plt.plot(weekly_returns.index, ew_cumulative, label='Equal-Weight', linewidth=2, linestyle='--')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Cumulative Return', fontsize=12)
plt.title('Risk-Adjusted Thompson Sampling', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Sharpe-adjusted Sharpe ratio:")
print(f"  {returns_sharpe.mean() / returns_sharpe.std() * np.sqrt(52):.2f}")

# YOUR TURN: Try other risk adjustments
# - Maximum drawdown constraint
# - Value-at-Risk penalty
# - Kelly criterion sizing

## Summary

**What you learned:**

1. **Gaussian Thompson Sampling** extends naturally from Bernoulli to continuous rewards
2. **Normal-Normal conjugacy** gives closed-form posterior updates for mean return
3. **Real commodity allocation** shows Thompson Sampling adapts to changing market conditions
4. **Posteriors concentrate** as data accumulates, but maintain small uncertainty for exploration

**Commodity trading insights:**

- **Weekly rebalancing fits Thompson Sampling** — Natural for batched feedback
- **Allocation shifts over time** — Responds to recent performance (adaptive)
- **Never 100% in one commodity** — Maintains exploration for regime changes
- **Risk adjustment matters** — Sharpe-based selection improves risk-adjusted returns

**Limitations of this approach:**

1. **Assumes known variance** — In reality, volatility is time-varying. Extension: Normal-Gamma prior on both μ and σ².
2. **Ignores context** — Doesn't use market features (VIX, term structure, seasonality). Module 3 fixes this with contextual bandits.
3. **No diversification constraint** — Goes 100% into one commodity. Real portfolios need risk limits.
4. **Stationarity assumption** — Long-run mean may shift (regime changes). Extension: Discounted Thompson Sampling (Module 6).

**Next module:**

Module 3 introduces **Contextual Bandits** — decisions that depend on features like volatility regime, term structure, and seasonality. You'll learn LinUCB and extend Thompson Sampling to condition on market state.